In [7]:
# Tools for Fundraising Support Assistant

from __future__ import annotations

from typing import Dict, Any, Optional, List
import pandas as pd
import numpy as np


# -----------------------------
# Helpers
# -----------------------------

def _as_month(d: str) -> pd.Timestamp:
    # Accept "YYYY-MM"
    return pd.to_datetime(d, format="%Y-%m")

def _month_range_end(as_of: pd.Timestamp) -> pd.Timestamp:
    # Normalize to first day of month
    return pd.Timestamp(year=as_of.year, month=as_of.month, day=1)

def _last_n_months(df: pd.DataFrame, as_of_month: pd.Timestamp, n: int) -> pd.DataFrame:
    start = as_of_month - pd.DateOffset(months=n-1)
    mask = (df["date"] >= start) & (df["date"] <= as_of_month)
    return df.loc[mask].copy()

def _ytd(df: pd.DataFrame, as_of_month: pd.Timestamp) -> pd.DataFrame:
    start = pd.Timestamp(year=as_of_month.year, month=1, day=1)
    mask = (df["date"] >= start) & (df["date"] <= as_of_month)
    return df.loc[mask].copy()

def _safe_div(a: float, b: float) -> Optional[float]:
    b = float(b)
    if b == 0:
        return None
    return float(a) / b


# -----------------------------
# Tool 1: Snapshot (MTD/YTD/TTM + breakdown)
# -----------------------------

def get_fundraising_snapshot(data,as_of, breakdown_prefix = "revenue_"):
    """
    Returns a simple snapshot as-of a month:
      - month totals (revenue, costs, net)
      - YTD totals
      - trailing-12-month (TTM) totals
      - optional revenue breakdown shares (revenue_* columns)
    """
    as_of_month = _month_range_end(_as_month(as_of))
    df = data

    row = df.loc[df["date"] == as_of_month]
    if row.empty:
        return {"ok": False, "error": f"No data for month {as_of}.", "as_of": as_of}

    row = row.iloc[0]
    m_rev = float(row["revenue_total"])
    m_cost = float(row["costs_total"])
    m_net = m_rev - m_cost

    ytd_df = _ytd(df, as_of_month)
    ttm_df = _last_n_months(df, as_of_month, 12)

    ytd_rev = float(ytd_df["revenue_total"].sum())
    ytd_cost = float(ytd_df["costs_total"].sum())
    ytd_net = ytd_rev - ytd_cost

    ttm_rev = float(ttm_df["revenue_total"].sum())
    ttm_cost = float(ttm_df["costs_total"].sum())
    ttm_net = ttm_rev - ttm_cost

    # Revenue breakdown (only if columns exist)
    breakdown_cols = [c for c in df.columns if c.startswith(breakdown_prefix) and c != "revenue_total"]
    breakdown = {}
    if breakdown_cols:
        month_break = {c: float(row[c]) for c in breakdown_cols}
        # Shares of total revenue (avoid division by zero)
        shares = {c: _safe_div(v, m_rev) for c, v in month_break.items()}
        breakdown = {"month": month_break, "month_share_of_total": shares}

    return {
        "ok": True,
        "as_of": as_of,
        "month": {"revenue": round(m_rev, 2), "costs": round(m_cost, 2), "net": round(m_net, 2)},
        "ytd": {"revenue": round(ytd_rev, 2), "costs": round(ytd_cost, 2), "net": round(ytd_net, 2)},
        "ttm": {"revenue": round(ttm_rev, 2), "costs": round(ttm_cost, 2), "net": round(ttm_net, 2)},
        "breakdown": breakdown,
    }


# -----------------------------
# Tool 2: YoY comparison (month + YTD)
# -----------------------------

def compare_yoy(data,as_of,metric = "revenue_total"):
    """
    Compares a metric to the same month last year and YTD vs prior-year YTD.
    metric examples: revenue_total, costs_total, revenue_individual, etc.
    """
    df = data
    as_of_month = _month_range_end(_as_month(as_of))
    prior = as_of_month - pd.DateOffset(years=1)

    if metric not in df.columns:
        return {"ok": False, "error": f"Unknown metric '{metric}'."}

    cur_row = df.loc[df["date"] == as_of_month]
    prv_row = df.loc[df["date"] == prior]
    if cur_row.empty or prv_row.empty:
        return {"ok": False, "error": f"Missing data for {as_of} or {prior.strftime('%Y-%m')}."}

    cur_val = float(cur_row.iloc[0][metric])
    prv_val = float(prv_row.iloc[0][metric])
    yoy_pct = None if prv_val == 0 else (cur_val - prv_val) / prv_val

    cur_ytd = float(_ytd(df, as_of_month)[metric].sum())
    prv_ytd = float(_ytd(df, prior)[metric].sum())
    ytd_yoy_pct = None if prv_ytd == 0 else (cur_ytd - prv_ytd) / prv_ytd

    return {
        "ok": True,
        "as_of": as_of,
        "metric": metric,
        "month": {
            "current": round(cur_val, 2),
            "prior_year": round(prv_val, 2),
            "yoy_pct": None if yoy_pct is None else round(yoy_pct, 4),
        },
        "ytd": {
            "current": round(cur_ytd, 2),
            "prior_year": round(prv_ytd, 2),
            "yoy_pct": None if ytd_yoy_pct is None else round(ytd_yoy_pct, 4),
        },
    }


# -----------------------------
# Tool 3: Cost effectiveness KPIs
# -----------------------------

def cost_effectiveness(data,as_of, window_months = 12):
    """
    Fundraising KPI tool:
      - cost_to_raise_1 = costs / revenue
      - net = revenue - costs
      - margin = net / revenue
    Computed over a trailing window (default TTM).
    """
    df = data
    as_of_month = _month_range_end(_as_month(as_of))
    w = _last_n_months(df, as_of_month, window_months)

    rev = float(w["revenue_total"].sum())
    cost = float(w["costs_total"].sum())
    net = rev - cost

    ctr = _safe_div(cost, rev)          # cost per $1 raised
    margin = _safe_div(net, rev)

    return {
        "ok": True,
        "as_of": as_of,
        "window_months": window_months,
        "revenue": round(rev, 2),
        "costs": round(cost, 2),
        "net": round(net, 2),
        "cost_to_raise_1": None if ctr is None else round(ctr, 4),
        "net_margin": None if margin is None else round(margin, 4),
    }


# -----------------------------
# Tool 4: Goal pacing
# -----------------------------

def goal_pacing(data,as_of,annual_goal):
    """
    Simple pacing tool (no forecasting):
      - how much raised YTD
      - remaining to goal
      - remaining months in year
      - required avg per remaining month
      - current YTD monthly avg
    """
    df = data
    as_of_month = _month_range_end(_as_month(as_of))

    ytd_df = _ytd(df, as_of_month)
    raised_ytd = float(ytd_df["revenue_total"].sum())

    remaining = float(annual_goal) - raised_ytd
    remaining = max(0.0, remaining)

    month_num = as_of_month.month
    remaining_months = 12 - month_num
    # If as_of is Dec, remaining months = 0
    required_per_month = None if remaining_months == 0 else remaining / remaining_months

    ytd_months = month_num
    current_avg = None if ytd_months == 0 else raised_ytd / ytd_months

    status = "on_track"
    if required_per_month is not None and current_avg is not None:
        status = "behind" if current_avg < required_per_month else "ahead"

    return {
        "ok": True,
        "as_of": as_of,
        "annual_goal": round(float(annual_goal), 2),
        "raised_ytd": round(raised_ytd, 2),
        "remaining_to_goal": round(remaining, 2),
        "remaining_months": int(remaining_months),
        "required_avg_per_remaining_month": None if required_per_month is None else round(required_per_month, 2),
        "current_ytd_monthly_avg": None if current_avg is None else round(current_avg, 2),
        "pace_status": status,
    }




In [15]:
import pandas as pd

data = pd.read_csv("C://Users//elizabeth.mohr//Dropbox//GBA 479//Spring A//Full Time//Assignments//Take Home Final//backup//northbridge_fundraising_history.csv")
data["date"] = pd.to_datetime(df["date"], format="%Y-%m")
data.head()

,date,revenue_total,revenue_individual,revenue_foundation,revenue_corporate,revenue_events,revenue_online,costs_total,donors_total,new_donors,retained_donors,avg_gift
0,2021-01-01,84913.44,39546.93,19466.42,7880.79,9451.67,8567.63,21350.09,465,46,419,103.47
1,2021-02-01,79066.61,36596.49,16246.94,11195.22,11250.64,3777.32,24143.93,453,39,414,89.13
2,2021-03-01,97786.78,42802.25,24718.76,11045.58,17362.62,1857.57,26948.07,372,47,325,120.05
3,2021-04-01,109145.97,53351.37,25021.48,12479.73,16045.39,2248.00,30188.78,423,70,353,131.44
4,2021-05-01,79308.97,34864.55,18140.59,12921.52,9785.44,3596.87,21993.90,411,37,374,93.58


In [17]:
# Example use: “Give me a quick snapshot as of Oct 2025.”
snap = get_fundraising_snapshot(data, as_of="2025-08")
snap

# Example fields you might display in the assistant:
# snap["month"] -> revenue/costs/net for the month
# snap["ytd"]   -> revenue/costs/net year-to-date
# snap["ttm"]   -> trailing-12-month totals
# snap["breakdown"] -> revenue category amounts + shares (if columns exist)

{'ok': True,
 'as_of': '2025-08',
 'month': {'revenue': 116844.74, 'costs': 26757.03, 'net': 90087.71},
 'ytd': {'revenue': 856562.76, 'costs': 222444.33, 'net': 634118.43},
 'ttm': {'revenue': 1317599.0, 'costs': 328541.35, 'net': 989057.65},
 'breakdown': {'month': {'revenue_individual': 54423.47,
   'revenue_foundation': 23845.71,
   'revenue_corporate': 13479.01,
   'revenue_events': 18612.43,
   'revenue_online': 6484.12},
  'month_share_of_total': {'revenue_individual': 0.4657759519170482,
   'revenue_foundation': 0.20408030348648984,
   'revenue_corporate': 0.11535829511880466,
   'revenue_events': 0.15929198010967374,
   'revenue_online': 0.05549346936798353}}}

In [18]:
# Use case: “How does revenue this month compare to the same month last year?”
yoy_rev = compare_yoy(data, as_of="2025-10", metric="revenue_total")
yoy_rev

# Example outputs:
# yoy_rev["month"]["yoy_pct"] -> percent change vs same month prior year
# yoy_rev["ytd"]["yoy_pct"]   -> percent change vs prior-year YTD

{'ok': True,
 'as_of': '2025-10',
 'metric': 'revenue_total',
 'month': {'current': 101666.55, 'prior_year': 91300.28, 'yoy_pct': 0.1135},
 'ytd': {'current': 1058865.4, 'prior_year': 952986.4, 'yoy_pct': 0.1111}}

In [19]:
# Use case: “What’s our cost to raise a dollar over the last 12 months?”
kpi_ttm = cost_effectiveness(data, as_of="2025-10", window_months=12)
kpi_ttm

# Example outputs:
# kpi_ttm["cost_to_raise_1"] -> costs/revenue
# kpi_ttm["net_margin"]      -> (revenue-costs)/revenue


{'ok': True,
 'as_of': '2025-10',
 'window_months': 12,
 'revenue': 1340672.72,
 'costs': 337628.81,
 'net': 1003043.91,
 'cost_to_raise_1': 0.2518,
 'net_margin': 0.7482}

In [20]:
# Use case: “We have a $1.2M annual goal. Are we on track as of Oct 2025?”
pacing = goal_pacing(data, as_of="2025-10", annual_goal=1_200_000)
pacing

# Example outputs:
# pacing["required_avg_per_remaining_month"]
# pacing["current_ytd_monthly_avg"]
# pacing["pace_status"] -> "ahead" / "behind" / "on_track"

{'ok': True,
 'as_of': '2025-10',
 'annual_goal': 1200000.0,
 'raised_ytd': 1058865.4,
 'remaining_to_goal': 141134.6,
 'remaining_months': 2,
 'required_avg_per_remaining_month': 70567.3,
 'current_ytd_monthly_avg': 105886.54,
 'pace_status': 'ahead'}